In [1]:
import pandas as pd

# === File paths ===
# Adjust to your local directory
file_yt_x = "../../../data/connect_survey_yt_x.csv"
file_ts_tt = "../../../data/connect_survey_ts_tt.csv"

# === Load datasets ===
yt_x = pd.read_csv(file_yt_x)
ts_tt = pd.read_csv(file_ts_tt)

# === Columns to inspect ===
demo_cols = [
    "Age", "Education", "Sex", "Occupation Field",
    "Relationship/Marital Status", "Political Party",
    "Ethnicity", "Gender", "Country Of Residence",
    "Household Income", "Race", "Employment Status"
]

# === Function to show unique values ===
def show_unique_values(df, label):
    print(f"\n\n{'='*40}\nUnique Values for {label}\n{'='*40}")
    for col in demo_cols:
        if col in df.columns:
            print(f"\n[{col}]")
            uniques = df[col].dropna().unique()
            print(uniques)
            print(f"→ Total unique: {len(uniques)}")

# === Display unique categories ===
show_unique_values(yt_x, "YouTube + X Survey")
show_unique_values(ts_tt, "Truth Social + TikTok Survey")

# === Inspect numeric Age values ===
def inspect_age(df, label):
    print(f"\n{'='*20}\nAge summary for {label}\n{'='*20}")
    if "Age" in df.columns:
        # Clean and convert to numeric
        df["Age_num"] = pd.to_numeric(df["Age"], errors="coerce")
        print(df["Age_num"].describe())
        print("Sample age values:", df["Age_num"].dropna().unique()[:20])

inspect_age(yt_x, "YouTube + X Survey")
inspect_age(ts_tt, "Truth Social + TikTok Survey")




Unique Values for YouTube + X Survey

[Age]
[41 37 36 24 32 27 42 40 61 21 43 26 29 71 35 20 28 48 63 31 60 22 49 65
 66 39 33 69 34 70 23 19 64 30 18 72 44 68 51 38 59 47 45 62 46 52 55 25
 67 76 77 74 73 58 83 56 75 78 81 79 54 57 50 53]
→ Total unique: 64

[Education]
["Bachelor's degree (for example: BA, AB, BS)"
 'Some college, but no degree' 'Less than a high school diploma'
 "Master's degree (for example: MA, MS, MEng, MEd, MSW, MBA)"
 'Professional degree (for example: MD, DDS, DVM, LLB, JD)'
 'High school graduate - high school diploma or the equivalent (for example: GED)'
 'Associate degree (for example: AA, AS)'
 'Doctorate degree (for example: PhD, EdD)' 'No formal education'
 'Prefer not to say']
→ Total unique: 10

[Sex]
['Male' 'Female' 'Prefer not to say']
→ Total unique: 3

[Occupation Field]
['Finance' 'Information Technology' 'Arts'
 'Science, Technology, Engineering & Mathematics' 'Medicine'
 'Manufacturing' 'Social Sciences' 'Other' 'I’d Rather Not Say'
 'Hospita

In [2]:
!openpyxl

zsh:1: command not found: openpyxl


In [3]:
import pandas as pd
import numpy as np

# === Load all data ===
yt_x = pd.read_csv("../../../data/connect_survey_yt_x.csv")
ts_tt = pd.read_csv("../../../data/connect_survey_ts_tt.csv")
survey_cleaned = pd.read_csv("../../../data/survey_cleaned.csv")
pairs_yt_x = pd.read_csv("../../../data/pairs_data_yt_x.csv")
pairs_ts_tt = pd.read_csv("../../../data/pairs_data_tt_ts.csv")

# === Clean up column names and ensure string IDs ===
for df in [yt_x, ts_tt, survey_cleaned, pairs_yt_x, pairs_ts_tt]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if "id" in col.lower():
            df[col] = df[col].astype(str).str.strip()

# === Merge logic ===
# Map connect_survey → response_id using survey_cleaned.csv
yt_x_merged = yt_x.merge(
    survey_cleaned[["participant_id", "response_id"]],
    left_on="ParticipantId", right_on="participant_id", how="inner"
)
ts_tt_merged = ts_tt.merge(
    survey_cleaned[["participant_id", "response_id"]],
    left_on="ParticipantId", right_on="participant_id", how="inner"
)

# Filter only response_ids that appear in pairs files
yt_x_filtered = yt_x_merged[yt_x_merged["response_id"].isin(pairs_yt_x["response_id"])]
ts_tt_filtered = ts_tt_merged[ts_tt_merged["response_id"].isin(pairs_ts_tt["response_id"])]

print(f"Matched participants (YouTube + X): {len(yt_x_filtered)}")
print(f"Matched participants (Truth Social + TikTok): {len(ts_tt_filtered)}")

# === Add age binning ===
age_bins = [17, 24, 34, 44, 54, 64, 74, 84, 100]
age_labels = ["18–24", "25–34", "35–44", "45–54", "55–64", "65–74", "75–84", "85+"]

def process_age(df):
    df["Age_num"] = pd.to_numeric(df["Age"], errors="coerce")
    df["Age Range"] = pd.cut(df["Age_num"], bins=age_bins, labels=age_labels, right=True)
    return df

yt_x_filtered = process_age(yt_x_filtered)
ts_tt_filtered = process_age(ts_tt_filtered)

# === Demographic summary ===
demo_cols = [
    "Age", "Education", "Sex", "Occupation Field",
    "Relationship/Marital Status", "Political Party",
    "Ethnicity", "Gender", "Country Of Residence",
    "Household Income", "Race", "Employment Status"
]

def demographic_summary(df, label):
    total = len(df)
    rows = []

    # Age
    age_counts = df["Age Range"].value_counts(dropna=False).sort_index()
    for age, count in age_counts.items():
        pct = round(count / total * 100, 1)
        rows.append(["Age Range", age, count, pct])

    # Other categories
    for col in demo_cols[1:]:
        if col in df.columns:
            counts = df[col].value_counts(dropna=False)
            for val, count in counts.items():
                pct = round(count / total * 100, 1)
                rows.append([col, val, count, pct])

    summary_df = pd.DataFrame(rows, columns=["Category", "Value", "Count", "Percent"])
    out_name = f"../../../output/tab_s19_source_{label.lower().replace(' ', '_')}.csv"
    summary_df.to_csv(out_name, index=False)
    print(f"Saved → {out_name}")
    return summary_df

yt_x_summary = demographic_summary(yt_x_filtered, "YouTube_X_final")
ts_tt_summary = demographic_summary(ts_tt_filtered, "TruthSocial_TikTok_final")


Matched participants (YouTube + X): 601
Matched participants (Truth Social + TikTok): 606
Saved → ../../../output/tab_s19_source_youtube_x_final.csv
Saved → ../../../output/tab_s19_source_truthsocial_tiktok_final.csv


In [4]:
import pandas as pd
import numpy as np

# === Load all data ===
yt_x = pd.read_csv("../../../data/connect_survey_yt_x.csv")
ts_tt = pd.read_csv("../../../data/connect_survey_ts_tt.csv")
survey_cleaned = pd.read_csv("../../../data/survey_cleaned.csv")
pairs_yt_x = pd.read_csv("../../../data/pairs_data_yt_x.csv")
pairs_ts_tt = pd.read_csv("../../../data/pairs_data_tt_ts.csv")

# === Clean up column names and ensure string IDs ===
for df in [yt_x, ts_tt, survey_cleaned, pairs_yt_x, pairs_ts_tt]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if "id" in col.lower():
            df[col] = df[col].astype(str).str.strip()

# === Merge logic ===
yt_x_merged = yt_x.merge(
    survey_cleaned[["participant_id", "response_id"]],
    left_on="ParticipantId", right_on="participant_id", how="inner"
)
ts_tt_merged = ts_tt.merge(
    survey_cleaned[["participant_id", "response_id"]],
    left_on="ParticipantId", right_on="participant_id", how="inner"
)

# === Matching + diagnostics ===
# total survey participants that had a response_id
ytx_total_mapped = len(yt_x_merged)
tstt_total_mapped = len(ts_tt_merged)

# matched to pairs
ytx_matched = yt_x_merged["response_id"].isin(pairs_yt_x["response_id"])
tstt_matched = ts_tt_merged["response_id"].isin(pairs_ts_tt["response_id"])

ytx_filtered = yt_x_merged[ytx_matched]
tstt_filtered = ts_tt_merged[tstt_matched]

# === Print match diagnostics ===
print(f"YouTube + X: total in survey file = {len(yt_x)}")
print(f"YouTube + X: successfully mapped to response_id = {ytx_total_mapped}")
print(f"YouTube + X: matched to pairs file = {ytx_matched.sum()}")
print(f"YouTube + X: NOT matched to pairs file = {(~ytx_matched).sum()}")
print("-" * 60)
print(f"Truth Social + TikTok: total in survey file = {len(ts_tt)}")
print(f"Truth Social + TikTok: successfully mapped to response_id = {tstt_total_mapped}")
print(f"Truth Social + TikTok: matched to pairs file = {tstt_matched.sum()}")
print(f"Truth Social + TikTok: NOT matched to pairs file = {(~tstt_matched).sum()}")
print("-" * 60)

# === Add age binning ===
age_bins = [17, 24, 34, 44, 54, 64, 74, 84, 100]
age_labels = ["18–24", "25–34", "35–44", "45–54", "55–64", "65–74", "75–84", "85+"]

def process_age(df):
    df["Age_num"] = pd.to_numeric(df["Age"], errors="coerce")
    df["Age Range"] = pd.cut(df["Age_num"], bins=age_bins, labels=age_labels, right=True)
    return df

ytx_filtered = process_age(ytx_filtered)
tstt_filtered = process_age(tstt_filtered)

# === Demographic summary ===
demo_cols = [
    "Age", "Education", "Sex", "Occupation Field",
    "Relationship/Marital Status", "Political Party",
    "Ethnicity", "Gender", "Country Of Residence",
    "Household Income", "Race", "Employment Status"
]

def demographic_summary(df, label):
    total = len(df)
    rows = []
    # Age first
    age_counts = df["Age Range"].value_counts(dropna=False).sort_index()
    for age, count in age_counts.items():
        pct = round(count / total * 100, 1)
        rows.append(["Age Range", age, count, pct])
    # Other variables
    for col in demo_cols[1:]:
        if col in df.columns:
            counts = df[col].value_counts(dropna=False)
            for val, count in counts.items():
                pct = round(count / total * 100, 1)
                rows.append([col, val, count, pct])
    summary_df = pd.DataFrame(rows, columns=["Category", "Value", "Count", "Percent"])
    out_name = f"../../../output/tab_s19_source_{label.lower().replace(' ', '_')}.csv"
    summary_df.to_csv(out_name, index=False)
    print(f"Saved → {out_name}")
    return summary_df

ytx_summary = demographic_summary(ytx_filtered, "YouTube_X_final")
tstt_summary = demographic_summary(tstt_filtered, "TruthSocial_TikTok_final")


YouTube + X: total in survey file = 621
YouTube + X: successfully mapped to response_id = 601
YouTube + X: matched to pairs file = 601
YouTube + X: NOT matched to pairs file = 0
------------------------------------------------------------
Truth Social + TikTok: total in survey file = 613
Truth Social + TikTok: successfully mapped to response_id = 606
Truth Social + TikTok: matched to pairs file = 606
Truth Social + TikTok: NOT matched to pairs file = 0
------------------------------------------------------------
Saved → ../../../output/tab_s19_source_youtube_x_final.csv
Saved → ../../../output/tab_s19_source_truthsocial_tiktok_final.csv
